# Task 1 — Article Type Classification

This notebook asks which image-only method should predict the 124 `articleType` labels. It follows the original experiment path: establish a scratch CNN control, test whether HOG models can compete, add mild data augmentation, test class weighting, compare all saved evidence, inspect failures, and freeze one model for Notebook 06.

The saved outputs document completed registered experiments. A fresh Run All performs smoke checks. Full and final runs are deliberate separate runs.

## 1. Problem and output

The input is one fashion image linked to one product `id`. The output is one label from the fixed 124-class `articleType` vocabulary. This task fills only the `articleType` column of the required `id,gender,articleType,season,usage` prediction file.

Model development uses labelled development images only. The CNN is trained from random starting weights, which means it learns from this dataset rather than importing pretrained knowledge. Pretrained models may be discussed as benchmarks, but they are not eligible as the submitted model.


## 2. EDA evidence

Before choosing a model, we measure the parts of this dataset that can change the design. The next cell counts development images, classes, grayscale files, unusual image shapes, and labels that are missing from some training folds. It then links each problem to a choice that will be tested. These choices are hypotheses, not winners.


In [1]:
from dataclasses import asdict

import pandas as pd

from fashion.config import DEVELOPMENT_CLASS_SUMMARY_CSV
from fashion.data.dataset import get_samples, load_label_maps, load_splits
from fashion.task1 import (
    build_task1_decision_evidence,
    build_task1_problem_profile,
)

TARGET = "articleType"
splits = load_splits()
task1_development = get_samples(splits, partition="development", target=TARGET)
article_type_map = load_label_maps()[TARGET]
article_type_classes = tuple(article_type_map["classes"])
class_summary = pd.read_csv(DEVELOPMENT_CLASS_SUMMARY_CSV, keep_default_na=False)
problem_profile = build_task1_problem_profile(splits, class_summary)

display(pd.DataFrame([asdict(problem_profile)]))
display(build_task1_decision_evidence(problem_profile))


,development_products,class_count,minimum_class_products,maximum_class_products,grayscale_images,unusual_geometry_images,classes_with_fold_warnings,classes_untrainable_in_any_fold
0,32773,124,1,5748,294,12,26,12


,evidence,problem,choice_to_test
0,12 development images differ from the usual 60...,stretching changes shape and centre crops can ...,aspect-preserving 60-by-80 white canvas
1,294 development images are grayscale,input channel formats are inconsistent,deterministic RGB conversion for every model f...
2,124 classes range from 1 to 5748 products,accuracy can hide failure on rare classes,fixed-class macro-F1 plus per-class evidence
3,12 classes are untrainable in at least one fold,some validation folds cannot represent every c...,fixed-class macro-F1 with zero-support classes...
4,product shape and edge detail may separate vis...,raw pixels may not provide a strong low-data b...,HOG with k-NN and linear SVM baselines
5,32773 development products support a small ben...,a large network can overfit a limited labelled...,small scratch CNN against the classical baselines
6,26 classes have rare-class fold warnings,rare classes see too few distinct training exa...,training-only augmentation for the scratch CNN


### Observation — the data is strongly imbalanced

The development set has 32,773 products across 124 classes, but class size ranges from 1 to 5,748 images. Twenty-six classes have fold warnings, and 12 classes are absent from at least one fold's training rows. This means high accuracy can be produced by learning common labels while ignoring rare ones, so fixed-label macro-F1 and per-class evidence are necessary.

There are also 294 grayscale images and 12 images with unusual geometry. These counts justify deterministic RGB conversion and shape-preserving padding instead of assuming every file has the same channel format and size.


## 3. Safety contract

`data/processed/splits.csv` is the only split. We never call `train_test_split` or make a second split. Development rows use the same five saved folds for every candidate, so model comparisons use the same validation products. Holdout and quarantine labels stay sealed until Notebook 06.

Only image pixels are model input. Product names, years, file size, and the other targets are excluded because they could act as shortcuts. Every physical training run is registered in `results/runs.csv`, including its split hash, settings, metrics, runtime, and artifact paths.


## 4. Evaluation

Each final candidate uses all five saved folds. In each run, four folds train the model and the remaining fold validates it. Repeating this five times gives every development product one out-of-fold prediction from a model that did not train on that product.

The main score is fixed-label macro-F1 across all 124 classes. Macro-F1 gives a rare class the same importance as a common class, and a class that cannot be learned in a fold still receives an honest zero. The five-fold mean shows typical performance, while the sample standard deviation shows stability across folds.

Pooled out-of-fold macro-F1, per-class F1, confusion pairs, validation loss, weighted F1, Top-1 accuracy, and Top-5 accuracy help explain the result. They support the judgement but do not hide a weak main score.


## 5. Candidate hypotheses

The investigation follows the original experiment order and changes one main factor at a time:

1. A small scratch CNN without augmentation is the neural control.
2. HOG with k-NN and linear SVM tests whether simpler hand-built shape features can compete.
3. The same CNN with mild augmentation tests whether small image changes reduce overfitting.
4. The augmented CNN with balanced class weights tests whether giving rare classes more loss weight improves macro-F1.

All candidates use the same saved folds and fixed label vocabulary. The results are reported after the process is defined, so the evidence leads to the decision rather than the decision shaping the experiment.

## 6. Controlled preprocessing

Every CNN first applies EXIF orientation, converts the image to RGB, and places it on a shape-preserving 60-by-80 white canvas. This avoids stretching unusual images and gives grayscale files the same three-channel format as colour files. Normalization values are fitted using only the training rows inside each fold.

The control adds no random image change. Mild augmentation is applied to training images only: horizontal flips, rotations up to 5 degrees, translations up to 5%, small scale changes, and small brightness and contrast changes. Validation images stay deterministic. The weighted-loss candidate keeps this same augmentation, so its comparison with the augmented unweighted CNN isolates the loss change.


In [2]:
from fashion.task1 import (
    DEFAULT_TASK1_PREPROCESSING,
    TASK1_CONTROL_PREPROCESSING,
    TASK1_BALANCED_WEIGHTED_CANDIDATE,
    TASK1_MILD_AUG_CANDIDATE,
    TASK1_NO_AUG_CANDIDATE,
    run_task1_classical_experiment,
    run_task1_experiment,
)

preprocessing_candidates = {
    "control_no_augmentation": TASK1_CONTROL_PREPROCESSING.to_dict(),
    "hypothesis_mild_augmentation": DEFAULT_TASK1_PREPROCESSING.to_dict(),
}
cnn_candidates = pd.DataFrame(
    [
        {
            "candidate_id": candidate.candidate_id,
            "preprocessing_id": candidate.preprocessing.preprocessing_id,
            "loss_id": candidate.loss.loss_id,
        }
        for candidate in (
            TASK1_NO_AUG_CANDIDATE,
            TASK1_MILD_AUG_CANDIDATE,
            TASK1_BALANCED_WEIGHTED_CANDIDATE,
        )
    ]
)

display(pd.DataFrame(preprocessing_candidates).T)
display(cnn_candidates)


,preprocessing_id,image_size,pad_color,horizontal_flip_probability,max_rotation_degrees,max_translation_fraction,scale_range,brightness_range,contrast_range
control_no_augmentation,task1_rgb_60x80_no_aug_v1,"(80, 60)","(255, 255, 255)",0.0,0.0,0.0,"(1.0, 1.0)","(1.0, 1.0)","(1.0, 1.0)"
hypothesis_mild_augmentation,task1_rgb_60x80_mild_aug_v1,"(80, 60)","(255, 255, 255)",0.5,5.0,0.05,"(0.95, 1.05)","(0.9, 1.1)","(0.9, 1.1)"


,candidate_id,preprocessing_id,loss_id
0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1
1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1
2,task1_cnn_mild_aug_balanced_weighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_balanced_class_weighted_v1


### Observation — the comparisons are controlled

The table confirms that every CNN uses the same image size, padding, architecture, folds, and label map. The first comparison changes only augmentation. The second keeps augmentation fixed and changes only the loss. This makes it possible to connect a score change to the tested choice instead of several settings changing at once.


## 7. Scratch CNN control

We begin with a small CNN trained from random weights. It uses deterministic RGB conversion, the shape-preserving 60-by-80 canvas, and ordinary unweighted cross-entropy. It has no random augmentation. This is the neural control: later CNN experiments keep the architecture fixed and change one training choice.

The next cell reads the completed five-fold evidence. It does not train a model.

In [ ]:
from fashion.config import TASK1_EVIDENCE_DIR

scratch_candidate_id = "task1_cnn_no_aug_unweighted_v1"
saved_cnn_folds = pd.read_csv(TASK1_EVIDENCE_DIR / "fold_metrics.csv", keep_default_na=False)
saved_cnn_comparison = pd.read_csv(TASK1_EVIDENCE_DIR / "comparison.csv", keep_default_na=False)
saved_cnn_oof = pd.read_csv(TASK1_EVIDENCE_DIR / "oof_metrics.csv", keep_default_na=False)

display(saved_cnn_folds.loc[saved_cnn_folds["candidate_id"].eq(scratch_candidate_id)])
display(saved_cnn_comparison.loc[saved_cnn_comparison["candidate_id"].eq(scratch_candidate_id)])
display(saved_cnn_oof.loc[saved_cnn_oof["candidate_id"].eq(scratch_candidate_id)])

run_id,fold,candidate_id,preprocessing_id,loss_id,macro_f1,weighted_f1,top1_accuracy,top5_accuracy,validation_loss
task1-cnn-task1_cnn_no_aug_unweighted_v1-f0-s2753-2b671c263f75,0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.498052,0.834539,0.837784,0.977873,1.072126
task1-cnn-task1_cnn_no_aug_unweighted_v1-f1-s2753-326863573f92,1,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.519482,0.844484,0.848231,0.978951,1.058708
task1-cnn-task1_cnn_no_aug_unweighted_v1-f2-s2753-4a2ad5b813b3,2,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.517600,0.826542,0.832138,0.973447,1.253813
task1-cnn-task1_cnn_no_aug_unweighted_v1-f3-s2753-cbd14501bc35,3,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.536994,0.839710,0.843302,0.978792,1.168490
task1-cnn-task1_cnn_no_aug_unweighted_v1-f4-s2753-15a400dd82cb,4,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.585240,0.845254,0.849474,0.980784,1.111062


candidate_id,preprocessing_id,loss_id,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,top1_accuracy_mean,top1_accuracy_std,top5_accuracy_mean,top5_accuracy_std
task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.531473,0.03307,0.838106,0.007759,0.842186,0.007267,0.977969,0.002739


candidate_id,preprocessing_id,loss_id,macro_f1_124,weighted_f1,top1_accuracy,top5_accuracy
task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.556926,0.839208,0.842187,0.97797


### Observation — the scratch CNN is a strong control

The first table shows one row for each held-out development fold. The second summarizes the five fold scores, and the third pools the OOF predictions so every development product is scored once by a model that did not train on it.

The scratch CNN reached 0.5315 ± 0.0331 mean macro-F1, 84.22% Top-1 accuracy, and 97.80% Top-5 accuracy. Its pooled OOF macro-F1 was 0.5569. This establishes a strong neural control, but the spread from 0.4981 to 0.5852 across the saved folds shows that the difficulty changes between data groups.

**Decision:** keep the scratch CNN as the control and test whether simpler HOG models can match it.

## 8. HOG baselines

HOG turns each image into edge-and-shape features. k-NN predicts from nearby training examples, while the linear SVM learns class boundaries in the HOG feature space. These experiments test whether a simpler hand-built representation can compete with the scratch CNN.

The saved outputs document completed experiments for the selected classical configurations. A fresh Run All performs a smoke check only. Each final condition used all five saved folds and wrote registered evidence.

In [8]:
CLASSICAL_STAGE = "smoke"  # Use "tune" and then "final" only as separate deliberate runs.
classic_experiment = run_task1_classical_experiment(
    splits,
    article_type_map,
    stage=CLASSICAL_STAGE,
)
display(classic_experiment.fold_metrics)
if not classic_experiment.tuning.empty:
    display(classic_experiment.tuning)
if not classic_experiment.comparison.empty:
    display(classic_experiment.comparison)
if not classic_experiment.oof_metrics.empty:
    display(classic_experiment.oof_metrics)


,run_id,fold,candidate_id,hog_id,model_family,macro_f1,weighted_f1,top1_accuracy,top5_accuracy
0,task1-classical-task1_gray_hog_ppc10_v1-knn-k3...,0,task1_gray_hog_ppc10_v1-knn-k3-distance,task1_gray_hog_ppc10_v1,task1_hog_knn_v1,0.489685,0.789193,0.798413,0.890279
1,task1-classical-task1_gray_hog_ppc10_v1-knn-k3...,1,task1_gray_hog_ppc10_v1-knn-k3-distance,task1_gray_hog_ppc10_v1,task1_hog_knn_v1,0.496920,0.782313,0.791489,0.898414
2,task1-classical-task1_gray_hog_ppc10_v1-knn-k3...,2,task1_gray_hog_ppc10_v1-knn-k3-distance,task1_gray_hog_ppc10_v1,task1_hog_knn_v1,0.496270,0.780333,0.790325,0.891958
3,task1-classical-task1_gray_hog_ppc10_v1-knn-k3...,3,task1_gray_hog_ppc10_v1-knn-k3-distance,task1_gray_hog_ppc10_v1,task1_hog_knn_v1,0.502577,0.786799,0.795545,0.899298
4,task1-classical-task1_gray_hog_ppc10_v1-knn-k3...,4,task1_gray_hog_ppc10_v1-knn-k3-distance,task1_gray_hog_ppc10_v1,task1_hog_knn_v1,0.525995,0.791549,0.801891,0.896142
5,task1-classical-task1_gray_hog_ppc10_v1-linear...,0,task1_gray_hog_ppc10_v1-linear-svm-c0.1-balanced,task1_gray_hog_ppc10_v1,task1_hog_linear_svm_v1,0.462375,0.782172,0.778117,0.946284
6,task1-classical-task1_gray_hog_ppc10_v1-linear...,1,task1_gray_hog_ppc10_v1-linear-svm-c0.1-balanced,task1_gray_hog_ppc10_v1,task1_hog_linear_svm_v1,0.502251,0.784686,0.779134,0.943868
7,task1-classical-task1_gray_hog_ppc10_v1-linear...,2,task1_gray_hog_ppc10_v1-linear-svm-c0.1-balanced,task1_gray_hog_ppc10_v1,task1_hog_linear_svm_v1,0.500543,0.772063,0.769113,0.938501
8,task1-classical-task1_gray_hog_ppc10_v1-linear...,3,task1_gray_hog_ppc10_v1-linear-svm-c0.1-balanced,task1_gray_hog_ppc10_v1,task1_hog_linear_svm_v1,0.483450,0.785299,0.781355,0.948886
9,task1-classical-task1_gray_hog_ppc10_v1-linear...,4,task1_gray_hog_ppc10_v1-linear-svm-c0.1-balanced,task1_gray_hog_ppc10_v1,task1_hog_linear_svm_v1,0.495666,0.777008,0.774592,0.945402


,candidate_id,hog_id,model_family,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,top1_accuracy_mean,top1_accuracy_std,top5_accuracy_mean,top5_accuracy_std
0,task1_gray_hog_ppc10_v1-knn-k3-distance,task1_gray_hog_ppc10_v1,task1_hog_knn_v1,0.502289,0.014018,0.786037,0.004673,0.795533,0.004801,0.895218,0.003960
1,task1_gray_hog_ppc10_v1-linear-svm-c0.1-balanced,task1_gray_hog_ppc10_v1,task1_hog_linear_svm_v1,0.488857,0.016529,0.780246,0.005622,0.776462,0.004778,0.944588,0.003859


,candidate_id,macro_f1,weighted_f1,top1_accuracy,top5_accuracy
0,task1_gray_hog_ppc10_v1-knn-k3-distance,0.526879,0.787157,0.795533,0.895219
1,task1_gray_hog_ppc10_v1-linear-svm-c0.1-balanced,0.505912,0.780849,0.776462,0.944589


### Observation — HOG is useful but weaker than the scratch CNN

The output first shows one row per fold for both selected HOG models, then a five-fold summary and pooled OOF scores. HOG with distance-weighted 3-neighbour k-NN reached 0.5023 ± 0.0140 mean macro-F1 and 79.55% Top-1 accuracy. The balanced linear SVM reached 0.4889 ± 0.0165 macro-F1 and 77.65% Top-1 accuracy.

KNN is the stronger classical single-label baseline because it has better macro-F1 and Top-1. SVM has much better Top-5 accuracy, 94.46% versus 89.52%, which means it ranks the correct class inside a short candidate list more often. Both remain below the scratch CNN on the main macro-F1 score.

**Decision:** keep KNN as the strongest classical baseline; neither HOG model replaces the scratch CNN.

## 9. Mild data augmentation

We now change one part of the scratch CNN: training images receive mild flips, rotations, translations, scale changes, and brightness and contrast changes. The architecture, folds, label map, loss, image size, and validation transform stay fixed. This tests whether small image changes reduce overfitting without changing the model itself.

The saved output below contains the plain and augmented CNN side by side.

In [9]:
RUN_MODE = "smoke"  # Full runs are deliberate separate runs for the ten registered unweighted CNN folds.
task1_experiment = run_task1_experiment(
    splits,
    article_type_map,
    mode=RUN_MODE,
)
display(task1_experiment.fold_metrics)
if not task1_experiment.comparison.empty:
    display(task1_experiment.comparison)
if not task1_experiment.oof_metrics.empty:
    display(task1_experiment.oof_metrics)


,run_id,fold,candidate_id,preprocessing_id,loss_id,macro_f1,weighted_f1,top1_accuracy,top5_accuracy,validation_loss
0,task1-cnn-task1_cnn_no_aug_unweighted_v1-f0-s2...,0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.498052,0.834539,0.837784,0.977873,1.072126
1,task1-cnn-task1_cnn_no_aug_unweighted_v1-f1-s2...,1,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.519482,0.844484,0.848231,0.978951,1.058708
2,task1-cnn-task1_cnn_no_aug_unweighted_v1-f2-s2...,2,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.517600,0.826542,0.832138,0.973447,1.253813
3,task1-cnn-task1_cnn_no_aug_unweighted_v1-f3-s2...,3,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.536994,0.839710,0.843302,0.978792,1.168490
4,task1-cnn-task1_cnn_no_aug_unweighted_v1-f4-s2...,4,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.585240,0.845254,0.849474,0.980784,1.111062
5,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f0-...,0,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.508068,0.834493,0.840836,0.979551,0.581812
6,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f1-...,1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.521633,0.843648,0.848688,0.984137,0.554290
7,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f2-...,2,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.523251,0.829862,0.836563,0.978483,0.679289
8,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f3-...,3,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.508891,0.834636,0.837962,0.980623,0.598044
9,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f4-...,4,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.547079,0.844106,0.850541,0.981851,0.604823


,candidate_id,preprocessing_id,loss_id,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,top1_accuracy_mean,top1_accuracy_std,top5_accuracy_mean,top5_accuracy_std
0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.531473,0.033070,0.838106,0.007759,0.842186,0.007267,0.977969,0.002739
1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.521784,0.015783,0.837349,0.006263,0.842918,0.006338,0.980929,0.002186


,candidate_id,preprocessing_id,loss_id,macro_f1_124,weighted_f1,top1_accuracy,top5_accuracy
0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.556926,0.839208,0.842187,0.977970
1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.540755,0.838582,0.842919,0.980929


### Observation — augmentation trades score for lower fold variation

The output first shows the ten fold rows, followed by one five-fold summary row and one pooled OOF row for each CNN. The plain CNN reached the higher mean macro-F1: 0.5315 ± 0.0331. Mild augmentation reached 0.5218 ± 0.0158, a drop of 0.0097. Their Top-1 accuracy is almost identical at 84.22% and 84.29%, while augmentation has slightly higher Top-5 accuracy at 98.09% versus 97.80%.

Augmentation reduced variation across the five saved data groups and lowered mean validation loss from about 1.13 to 0.60. The lower loss shows better probability loss on these folds, but it does not prove that confidence is calibrated. The pooled OOF macro-F1 also favours the plain CNN, 0.5569 versus 0.5408.

**Decision:** the augmentation hypothesis does not pass its macro-F1 improvement rule. Keep it as a trade-off candidate because its fold variation and validation loss are lower.

## 10. Balanced class-weighted loss

This test keeps the mild-augmentation CNN and changes only the training loss. For each class present in a fold, `weight = training rows / (present classes × class rows)`. A rare class therefore contributes more to the loss than a common class. We calculate weights from that fold's training rows only, so validation information cannot leak into training. A class absent from the training fold receives weight zero because it cannot be learned there.

Validation loss remains unweighted, which keeps it comparable across the three CNN candidates. Full inverse-frequency weighting can push too hard toward extremely rare classes, so we check macro-F1, common-class performance, accuracy, and fold stability before accepting it. The saved outputs document the completed five weighted folds; a fresh Run All performs smoke checks, while a full run is a deliberate separate run.


In [11]:
from fashion.task1 import run_task1_weighted_experiment

WEIGHTED_MODE = "smoke"  # Full runs are deliberate separate runs for the five new weighted folds.
weighted_experiment = run_task1_weighted_experiment(
    splits,
    article_type_map,
    mode=WEIGHTED_MODE,
)
display(weighted_experiment.fold_metrics)
if not weighted_experiment.comparison.empty:
    display(weighted_experiment.comparison)
    display(weighted_experiment.oof_metrics)


,run_id,fold,candidate_id,preprocessing_id,loss_id,macro_f1,weighted_f1,top1_accuracy,top5_accuracy,validation_loss
0,task1-cnn-task1_cnn_no_aug_unweighted_v1-f0-s2...,0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.498052,0.834539,0.837784,0.977873,1.072126
1,task1-cnn-task1_cnn_no_aug_unweighted_v1-f1-s2...,1,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.519482,0.844484,0.848231,0.978951,1.058708
2,task1-cnn-task1_cnn_no_aug_unweighted_v1-f2-s2...,2,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.517600,0.826542,0.832138,0.973447,1.253813
3,task1-cnn-task1_cnn_no_aug_unweighted_v1-f3-s2...,3,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.536994,0.839710,0.843302,0.978792,1.168490
4,task1-cnn-task1_cnn_no_aug_unweighted_v1-f4-s2...,4,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.585240,0.845254,0.849474,0.980784,1.111062
5,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f0-...,0,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.508068,0.834493,0.840836,0.979551,0.581812
6,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f1-...,1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.521633,0.843648,0.848688,0.984137,0.554290
7,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f2-...,2,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.523251,0.829862,0.836563,0.978483,0.679289
8,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f3-...,3,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.508891,0.834636,0.837962,0.980623,0.598044
9,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f4-...,4,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.547079,0.844106,0.850541,0.981851,0.604823


,candidate_id,preprocessing_id,loss_id,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,top1_accuracy_mean,top1_accuracy_std,top5_accuracy_mean,top5_accuracy_std
0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.531473,0.033070,0.838106,0.007759,0.842186,0.007267,0.977969,0.002739
1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.521784,0.015783,0.837349,0.006263,0.842918,0.006338,0.980929,0.002186
2,task1_cnn_mild_aug_balanced_weighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_balanced_class_weighted_v1,0.456413,0.018133,0.713878,0.024425,0.691510,0.024893,0.952491,0.007479


,candidate_id,preprocessing_id,loss_id,macro_f1_124,weighted_f1,top1_accuracy,top5_accuracy
0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.556926,0.839208,0.842187,0.977970
1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.540755,0.838582,0.842919,0.980929
2,task1_cnn_mild_aug_balanced_weighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_balanced_class_weighted_v1,0.466383,0.714510,0.691514,0.952491


### Observation — full balancing is too aggressive

The output shows all fifteen CNN fold rows, then the three-candidate five-fold summary and pooled OOF table. The balanced weighted CNN reached 0.4564 ± 0.0181 mean macro-F1, 69.15% Top-1 accuracy, and 0.7139 weighted F1. Every main measure is below the augmented unweighted CNN, and average validation loss rises to about 1.11.

Weighting reduces the number of zero-F1 classes only from 21 to 20, which is too small a gain to justify the damage across the remaining classes. Rare classes with very few examples receive large influence but still lack enough distinct images to learn reliable patterns. This is a likely explanation, not proof of one cause.

**Decision:** reject full inverse-frequency class weighting for this CNN.

## 11. Combined five-fold and OOF comparison

This section reads the saved Task 1 evidence rather than copying scores by hand. `fold_metrics.csv` contains one row per CNN fold, `comparison.csv` summarizes the five-fold mean and standard deviation, and `oof_metrics.csv` scores the pooled out-of-fold predictions. Pooled OOF macro-F1 can differ from the mean of fold macro-F1 because it scores all development predictions together.

The following figure-writing cell saves the comparison and confusion matrices in `results/figures/task1/` for the final report. Each saved claim remains traceable to registered runs and evidence files.


In [13]:
from fashion.config import TASK1_EVIDENCE_DIR

for evidence_name in ("fold_metrics", "comparison", "oof_metrics"):
    evidence_path = TASK1_EVIDENCE_DIR / f"{evidence_name}.csv"
    if evidence_path.is_file():
        print(evidence_name)
        display(pd.read_csv(evidence_path, keep_default_na=False))
    else:
        print(f"{evidence_name} is not ready; complete the needed full runs first.")


fold_metrics


,run_id,fold,candidate_id,preprocessing_id,loss_id,macro_f1,weighted_f1,top1_accuracy,top5_accuracy,validation_loss
0,task1-cnn-task1_cnn_no_aug_unweighted_v1-f0-s2...,0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.498052,0.834539,0.837784,0.977873,1.072126
1,task1-cnn-task1_cnn_no_aug_unweighted_v1-f1-s2...,1,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.519482,0.844484,0.848231,0.978951,1.058708
2,task1-cnn-task1_cnn_no_aug_unweighted_v1-f2-s2...,2,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.517600,0.826542,0.832138,0.973447,1.253813
3,task1-cnn-task1_cnn_no_aug_unweighted_v1-f3-s2...,3,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.536994,0.839710,0.843302,0.978792,1.168490
4,task1-cnn-task1_cnn_no_aug_unweighted_v1-f4-s2...,4,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.585240,0.845254,0.849474,0.980784,1.111062
5,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f0-...,0,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.508068,0.834493,0.840836,0.979551,0.581812
6,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f1-...,1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.521633,0.843648,0.848688,0.984137,0.554290
7,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f2-...,2,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.523251,0.829862,0.836563,0.978483,0.679289
8,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f3-...,3,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.508891,0.834636,0.837962,0.980623,0.598044
9,task1-cnn-task1_cnn_mild_aug_unweighted_v1-f4-...,4,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.547079,0.844106,0.850541,0.981851,0.604823


comparison


,candidate_id,preprocessing_id,loss_id,macro_f1_mean,macro_f1_std,weighted_f1_mean,weighted_f1_std,top1_accuracy_mean,top1_accuracy_std,top5_accuracy_mean,top5_accuracy_std
0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.531473,0.033070,0.838106,0.007759,0.842186,0.007267,0.977969,0.002739
1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.521784,0.015783,0.837349,0.006263,0.842918,0.006338,0.980929,0.002186
2,task1_cnn_mild_aug_balanced_weighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_balanced_class_weighted_v1,0.456413,0.018133,0.713878,0.024425,0.691510,0.024893,0.952491,0.007479


oof_metrics


,candidate_id,preprocessing_id,loss_id,macro_f1_124,weighted_f1,top1_accuracy,top5_accuracy
0,task1_cnn_no_aug_unweighted_v1,task1_rgb_60x80_no_aug_v1,cross_entropy_unweighted_v1,0.556926,0.839208,0.842187,0.977970
1,task1_cnn_mild_aug_unweighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_unweighted_v1,0.540755,0.838582,0.842919,0.980929
2,task1_cnn_mild_aug_balanced_weighted_v1,task1_rgb_60x80_mild_aug_v1,cross_entropy_balanced_class_weighted_v1,0.466383,0.714510,0.691514,0.952491


In [14]:
from fashion.config import TASK1_FIGURE_DIR
from fashion.task1 import write_task1_comparison_figure, write_task1_confusion_figure

if WEIGHTED_MODE == "full" and not weighted_experiment.fold_metrics.empty:
    write_task1_comparison_figure(weighted_experiment.fold_metrics)
    for candidate_id, predictions in weighted_experiment.oof_predictions.items():
        write_task1_confusion_figure(
            predictions,
            article_type_classes,
            output=TASK1_FIGURE_DIR / f"cnn_oof_confusion_{candidate_id}.png",
        )
elif RUN_MODE == "full" and not task1_experiment.fold_metrics.empty:
    write_task1_comparison_figure(task1_experiment.fold_metrics)
    for candidate_id, predictions in task1_experiment.oof_predictions.items():
        write_task1_confusion_figure(
            predictions,
            article_type_classes,
            output=TASK1_FIGURE_DIR / f"cnn_oof_confusion_{candidate_id}.png",
        )

if CLASSICAL_STAGE == "final":
    for candidate_id, predictions in classic_experiment.oof_predictions.items():
        write_task1_confusion_figure(
            predictions,
            article_type_classes,
            output=TASK1_FIGURE_DIR / f"classical_oof_confusion_{candidate_id}.png",
        )


### Observation — the best mean and the lower-variation choice differ

The fold table contains one validation result per physical run. The comparison table summarizes the mean and sample standard deviation across the five saved folds. The pooled OOF table instead calculates one score after combining every development prediction. These values differ because averaging five fold scores is not the same calculation as scoring all predictions together.

By mean macro-F1, the order is plain CNN (0.5315), augmented CNN (0.5218), HOG KNN (0.5023), HOG SVM (0.4889), then weighted CNN (0.4564). The pooled OOF CNN results tell the same broad story: 0.5569, 0.5408, and 0.4664. Fold standard deviation describes differences across these five saved data groups; it does not prove repeat-run or random-seed stability.

The plain CNN is the main-metric winner. The augmented CNN has lower fold variation and lower validation loss. It has 475,516 parameters, a checkpoint of about 1.82 MB, and an average recorded fold training time of about 18.6 minutes on the project GPU.

**Decision:** carry the augmented CNN forward only as an explicit practical trade-off, not as the macro-F1 winner.

## 12. Learning-curve diagnosis

A final score does not show when learning improved or when overfitting began. The learning curves therefore average the five histories for each CNN across 20 epochs. HOG, k-NN, and SVM are not shown because they do not learn through the same epoch-by-epoch process.

On the loss chart, dashed lines are training loss and solid lines are validation loss; lower is better. On the macro-F1 chart, higher is better. Macro-F1 is shown for validation because it measures performance on unseen fold data, while training loss is the quantity used to update the CNN. The shaded bands show how much the five folds differ.


In [12]:
from collections import defaultdict
from pathlib import Path

from fashion.config import ROOT, TASK1_EVIDENCE_DIR, TASK1_FIGURE_DIR
from fashion.task1 import write_task1_learning_curve_figure
from fashion.train.registry import RunRegistry

expected_learning_candidates = {
    TASK1_NO_AUG_CANDIDATE.candidate_id,
    TASK1_MILD_AUG_CANDIDATE.candidate_id,
    TASK1_BALANCED_WEIGHTED_CANDIDATE.candidate_id,
}
fold_metrics_path = TASK1_EVIDENCE_DIR / "fold_metrics.csv"
learning_curve_histories = defaultdict(list)

if not fold_metrics_path.is_file():
    print("Learning curves are not ready; complete weighted full first.")
else:
    combined_folds = pd.read_csv(fold_metrics_path, keep_default_na=False)
    required_columns = {"run_id", "candidate_id", "fold"}
    has_required_columns = required_columns.issubset(combined_folds.columns)
    has_complete_folds = False
    if has_required_columns:
        fold_counts = combined_folds.groupby("candidate_id")["fold"].nunique()
        has_complete_folds = (
            len(combined_folds) == 15
            and set(combined_folds["candidate_id"]) == expected_learning_candidates
            and set(fold_counts) == {5}
        )

    if not has_required_columns or not has_complete_folds:
        print("Learning curves are not ready; complete weighted full first.")
    else:
        registry_rows = RunRegistry().read().set_index("run_id", drop=False)
        missing_history = []
        for row in combined_folds.sort_values(["candidate_id", "fold"]).itertuples(index=False):
            if row.run_id not in registry_rows.index:
                missing_history.append(str(row.run_id))
                continue
            registry_row = registry_rows.loc[row.run_id]
            history_path = Path(str(registry_row["history_path"]))
            if not history_path.is_absolute():
                history_path = ROOT / history_path
            if not history_path.is_file():
                missing_history.append(str(row.run_id))
                continue
            learning_curve_histories[str(row.candidate_id)].append(pd.read_csv(history_path))

        if missing_history or any(len(items) != 5 for items in learning_curve_histories.values()):
            print("Learning curves are not ready; complete weighted full first.")
        else:
            learning_curve_path = write_task1_learning_curve_figure(
                dict(learning_curve_histories),
                output=TASK1_FIGURE_DIR / "cnn_learning_curves.png",
            )
            print(f"Wrote {learning_curve_path}")




Wrote C:\Users\Khoa\Documents\MLA2\results\figures\task1\cnn_learning_curves.png


### Observation — the plain CNN overfits more

![Mean CNN learning curves across five folds](../results/figures/task1/cnn_learning_curves.png)

For the plain CNN, training loss keeps falling toward zero while validation loss stops improving and rises after roughly epoch 11. This widening gap is evidence of overfitting: the model becomes better at the images it saw but not at unseen fold images.

Mild augmentation keeps validation loss near 0.60 and its macro-F1 curve is steadier near 0.52. The weighted CNN finishes with a much lower validation macro-F1. Its dashed training-loss line uses a different weighted objective, so its absolute training-loss value should not be compared directly with the two unweighted dashed lines.


## 13. Weak-class/confusion analysis

One overall score can hide which labels fail. The weak-class table sorts classes with low F1 and shows their support, which is the number of true examples. The confusion-pair table counts common wrong label pairs and gives `example_ids` for later image inspection.

The weighted controller also returns the two earlier CNN candidates, so some prefixed rows repeat the same unweighted evidence. They are repeated views, not extra trained model types. Failure evidence explains risks and supports the decision, but it does not replace the fixed five-fold comparison.


In [15]:
from fashion.task1 import (
    build_task1_confusion_pairs,
    build_task1_weak_class_table,
)

per_class_evidence = {
    **{f"cnn:{key}": value for key, value in task1_experiment.per_class.items()},
    **{f"weighted:{key}": value for key, value in weighted_experiment.per_class.items()},
    **{f"classic:{key}": value for key, value in classic_experiment.per_class.items()},
}
oof_prediction_evidence = {
    **{f"cnn:{key}": value for key, value in task1_experiment.oof_predictions.items()},
    **{f"weighted:{key}": value for key, value in weighted_experiment.oof_predictions.items()},
    **{f"classic:{key}": value for key, value in classic_experiment.oof_predictions.items()},
}

if per_class_evidence:
    display(build_task1_weak_class_table(per_class_evidence, limit=10))
else:
    print("Weak-class evidence is not ready; complete full/final runs first.")

if oof_prediction_evidence:
    display(build_task1_confusion_pairs(oof_prediction_evidence, limit=10))
else:
    print("Confusion-pair evidence is not ready; complete full/final runs first.")


,candidate_id,class_index,class_name,support,precision,recall,f1
0,cnn:task1_cnn_no_aug_unweighted_v1,7,Body Wash and Scrub,1,0.0,0.0,0.0
1,cnn:task1_cnn_no_aug_unweighted_v1,23,Cushion Covers,1,0.0,0.0,0.0
2,cnn:task1_cnn_no_aug_unweighted_v1,44,Ipad,1,0.0,0.0,0.0
3,cnn:task1_cnn_no_aug_unweighted_v1,51,Key chain,1,0.0,0.0,0.0
4,cnn:task1_cnn_no_aug_unweighted_v1,64,Lounge Tshirts,1,0.0,0.0,0.0
...,...,...,...,...,...,...,...
65,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,76,Rain Jacket,1,0.0,0.0,0.0
66,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,77,Rain Trousers,1,0.0,0.0,0.0
67,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,90,Shoe Laces,1,0.0,0.0,0.0
68,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,106,Ties and Cufflinks,1,0.0,0.0,0.0


,candidate_id,true_label,predicted_label,error_count,example_ids
0,cnn:task1_cnn_no_aug_unweighted_v1,Sports Shoes,Casual Shoes,286,"1548,1549,1653"
1,cnn:task1_cnn_no_aug_unweighted_v1,Casual Shoes,Sports Shoes,281,"1543,1544,1545"
2,cnn:task1_cnn_no_aug_unweighted_v1,Tshirts,Tops,246,"1570,1763,1984"
3,cnn:task1_cnn_no_aug_unweighted_v1,Tops,Tshirts,235,"2120,2294,2700"
4,cnn:task1_cnn_no_aug_unweighted_v1,Flats,Heels,141,"2609,2613,2614"
...,...,...,...,...,...
65,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,Casual Shoes,Formal Shoes,166,"1917,2368,2371"
66,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,Flats,Heels,140,"2625,2626,2628"
67,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,Heels,Flats,128,"2878,2885,2888"
68,classic:task1_gray_hog_ppc10_v1-linear-svm-c0....,Tshirts,Innerwear Vests,104,"1966,2002,2011"


### Observation — rare labels and similar products remain the main risk

![OOF confusion matrix for the selected augmented unweighted CNN](../results/figures/task1/cnn_oof_confusion_task1_cnn_mild_aug_unweighted_v1.png)

The full normalized matrix keeps all 124 classes visible. Dark diagonal cells are correct predictions. Off-diagonal cells show where a true class is assigned to another label. This complete view proves the fixed label set was retained, but it is too dense to explain individual mistakes clearly.

The selected augmented CNN has 21 of 124 classes with pooled OOF F1 equal to zero. Many have only one or two true examples. Its largest directed errors include Casual Shoes → Sports Shoes (350), Tshirts → Tops (282), Sports Shoes → Casual Shoes (243), Tops → Tshirts (206), and Flats → Heels (169).

The next figures provide a closer look at the most common confusion pairs and example images.

In [ ]:
from fashion.config import ROOT, TASK1_EVIDENCE_DIR, TASK1_FIGURE_DIR
from fashion.task1 import (
    build_task1_confusion_detail,
    load_task1_oof_predictions,
    write_task1_confusion_example_figure,
    write_task1_confusion_pair_figure,
    write_task1_focused_confusion_figure,
)
from fashion.train.artifacts import atomic_write_csv
from fashion.train.registry import RunRegistry

selected_candidate_id = "task1_cnn_mild_aug_unweighted_v1"
saved_fold_metrics = pd.read_csv(TASK1_EVIDENCE_DIR / "fold_metrics.csv", keep_default_na=False)
selected_oof_predictions = load_task1_oof_predictions(
    saved_fold_metrics,
    RunRegistry().read(),
    candidate_id=selected_candidate_id,
    expected_ids=task1_development["id"].astype(int).tolist(),
)
confusion_detail = build_task1_confusion_detail(
    selected_oof_predictions,
    candidate_id=selected_candidate_id,
    limit=10,
)
atomic_write_csv(TASK1_EVIDENCE_DIR / "top_confusion_pairs.csv", confusion_detail)
write_task1_confusion_pair_figure(confusion_detail)
write_task1_focused_confusion_figure(selected_oof_predictions, confusion_detail)
write_task1_confusion_example_figure(
    selected_oof_predictions,
    splits,
    confusion_detail,
)
display(confusion_detail)

rank,candidate_id,true_label,predicted_label,error_count,true_support,error_rate,example_ids
1,task1_cnn_mild_aug_unweighted_v1,Casual Shoes,Sports Shoes,350,2276,0.153779,"1545,1546,1547"
2,task1_cnn_mild_aug_unweighted_v1,Tshirts,Tops,282,5748,0.049061,"1570,1763,1765"
3,task1_cnn_mild_aug_unweighted_v1,Sports Shoes,Casual Shoes,243,1691,0.143702,"1548,1549,1728"
4,task1_cnn_mild_aug_unweighted_v1,Tops,Tshirts,206,1369,0.150475,"2120,2294,2700"
5,task1_cnn_mild_aug_unweighted_v1,Flats,Heels,169,356,0.474719,"2626,2627,2628"
6,task1_cnn_mild_aug_unweighted_v1,Sandals,Flip Flops,96,735,0.130612,"2634,2636,7771"
7,task1_cnn_mild_aug_unweighted_v1,Formal Shoes,Casual Shoes,86,517,0.166344,"2377,2642,2824"
8,task1_cnn_mild_aug_unweighted_v1,Heels,Flats,79,890,0.088764,"2872,2890,3137"
9,task1_cnn_mild_aug_unweighted_v1,Shirts,Tshirts,71,2619,0.027110,"2050,2104,2125"
10,task1_cnn_mild_aug_unweighted_v1,Casual Shoes,Formal Shoes,63,2276,0.027680,"2371,2374,2376"


### A closer look at the largest mistakes

![Top ten directed confusion pairs](../results/figures/task1/top_confusion_pairs.png)

The bar chart uses error counts, so common classes can appear high simply because they have more products. The focused matrix below adds each true class's full support. For example, Casual Shoes → Sports Shoes is the largest count at 350 errors, equal to 15.4% of Casual Shoes. Flats → Heels has fewer errors at 169 but affects 47.5% of all Flats, so it is the more severe within-class failure.

![Focused confusion matrix for the largest pairs](../results/figures/task1/focused_confusion_matrix.png)

Each focused-matrix cell shows its count and its percentage of the true-label row. `Other` keeps predictions outside the eight displayed classes, so rows still describe all OOF examples for that class.

![Representative images from the largest confusion pairs](../results/figures/task1/confusion_examples.png)

The source images show why these errors are plausible. The confused shoe types share similar side profiles, while `Tshirts` and `Tops` overlap in sleeve length and overall shape. These examples support the visual-similarity explanation, but they also show a label-boundary problem: some business categories are difficult to separate from pixels alone.

**Decision:** report both error count and within-class rate, and require human review for rare or visually overlapping article types.

## 14. Development decision and Notebook 06 handoff

The no-augmentation CNN achieved the highest five-fold mean macro-F1: 0.5315 ± 0.0331. The mildly augmented unweighted CNN achieved a slightly lower macro-F1 of 0.5218 ± 0.0158.

Therefore, the original augmentation hypothesis did not pass its macro-F1 improvement rule. However, the augmented model had about 52% less fold variation, steadier validation loss, and less visible overfitting. We select it for final evaluation because we prefer more reliable performance across different data groups. This is a practical lower-fold-variation decision, not a claim that augmentation produced the highest score or has proven repeat-run stability.

The balanced class-weighted CNN was rejected because its mean macro-F1 fell to 0.4564.

### Frozen Task 1 handoff

- Output: `articleType`
- Selected candidate: `task1_cnn_mild_aug_unweighted_v1`
- Model family: `task1_small_cnn_v1`
- Preprocessing: `task1_rgb_60x80_mild_aug_v1`
- Loss: `cross_entropy_unweighted_v1`
- Fixed metric: macro-F1 over all 124 classes
- Five-fold result: 0.5218 ± 0.0158
- Evidence: `results/evidence/task1/comparison.csv`
- Fold evidence: `results/evidence/task1/fold_metrics.csv`
- OOF evidence: `results/evidence/task1/oof_metrics.csv`

Notebook 06 must refit this frozen configuration from scratch using all allowed development rows. It must then lock the checkpoint before opening the internal holdout. The holdout result must not be used to change the selected model or its settings.
